# CogVideoX-5B I2V Direct Inference (No Server)

在 Colab notebook cell 里直接运行 CogVideoX 推理，不启动 ComfyUI Server，
不跑 cloudflare tunnel。输出保存到 Google Drive。

**用法：** 从上到下依次运行每个 cell。首次运行会下载 ~2GB 模型，较慢。

## 1. 安装依赖

In [ ]:
!pip install -q diffusers transformers accelerate torchvision opencv-python pillow

## 2. 挂载 Google Drive

把模板图放在 Google Drive 中，输出也存到 Drive 上（Colab 重启不丢数据）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 设置路径

修改 `IMAGE_PATH` 为你的模板图路径。修改 `OUTPUT_DIR` 为输出目录。

In [ ]:
# ==== 请根据实际情况修改 ====
IMAGE_PATH = "/content/drive/MyDrive/character_template.png"
OUTPUT_DIR  = "/content/drive/MyDrive/cogvideo_outputs"
PROMPT = "a cute anime mother holding her baby, gentle horizontal sway, warm pastel colors, simple background"
NEGATIVE_PROMPT = "ugly, blurry, deformed, text, watermark, complex background, realistic"
SEED = 42
NUM_FRAMES = 25          # 25 帧 ≈ 3 秒 (@8fps)
NUM_STEPS = 25
GUIDANCE_SCALE = 6.0
# ===========================

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. 检查 GPU

In [ ]:
!nvidia-smi

## 5. 加载模型

关键优化：
- `torch.bfloat16` — 半精度，省显存
- `enable_sequential_cpu_offload` — 部分层放 CPU，防 OOM
- `vae.enable_tiling` — VAE 分块处理，省显存

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
import time

print("Loading model...")
t0 = time.time()

pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    torch_dtype=torch.bfloat16
)
pipe.enable_sequential_cpu_offload()  # 防 OOM：部分层放 CPU
pipe.vae.enable_tiling()              # VAE 分块处理

print(f"Model loaded in {time.time()-t0:.1f}s")

## 6. 运行推理

读模板图 → 跑 I2V → 输出 MP4

In [ ]:
print(f"Loading image: {IMAGE_PATH}")
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Image size: {image.size}")

print(f"\nGenerating {NUM_FRAMES} frames...")
print(f"Prompt: {PROMPT}")
t0 = time.time()

generator = torch.Generator(device="cpu").manual_seed(SEED)

video_frames = pipe(
    prompt=PROMPT,
    image=image,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE_SCALE,
    num_inference_steps=NUM_STEPS,
    generator=generator,
    negative_prompt=NEGATIVE_PROMPT,
).frames[0]

print(f"Inference done in {time.time()-t0:.1f}s ({len(video_frames)} frames)")

## 7. 保存视频

保存到 Google Drive，FPS=8（25 帧 ≈ 3 秒）

In [ ]:
output_file = os.path.join(OUTPUT_DIR, f"cogvideo_seed{SEED}_{NUM_FRAMES}f.mp4")
export_to_video(video_frames, output_file, fps=8)
print(f"\nSaved: {output_file}")
print(f"Size: {os.path.getsize(output_file) / 1024 / 1024:.1f} MB")

## 8. (可选) 预览视频

在 Colab 内直接播放

In [ ]:
from IPython.display import Video
Video(output_file, width=480)